# 🌊 Challenge A: Resilient Release Scheduling — Falcon Reservoir
## OQI Quantum Hackathon · Junio 2026

---

Este notebook implementa la solución completa del **Challenge A** del hackathon:

1. **Carga de datos** IBWC reales (CSVs de la presa Falcon)
2. **Baselines clásicos**: Historical Replay, Threshold Rule, Adaptive Rule
3. **Simulated Annealing clásico** como baseline fuerte
4. **Formulación QUBO** con one-hot encoding
5. **Solver QUBO** con SA de dimod (proxy cuántico) + reparación de factibilidad
6. **Gráficas y análisis** de resultados

> **Objetivo:** Encontrar una política de liberación de agua que maximice el *Storage Resilience Score* (SRS) del embalse Falcon durante un período de 26 semanas.

## 0. Instalación de Dependencias

In [ ]:
!pip install -q dimod dwave-neal

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
import matplotlib.pyplot as plt
import dimod
import neal

warnings.filterwarnings('ignore')
np.random.seed(42)

# Estilo de gráficas oscuro (estilo GitHub Dark)
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.alpha': 0.6,
    'font.size': 11,
    'axes.titlesize': 14,
    'figure.titlesize': 18,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'legend.labelcolor': '#c9d1d9',
})

COLORS = {
    'hist': '#8b949e', 'rule': '#f0883e', 'adapt': '#a371f7',
    'sa': '#58a6ff', 'qubo': '#3fb950', 'smin': '#f85149',
    'smax': '#1f6feb', 'accent': '#d2a8ff',
}

print('✅ Dependencias cargadas correctamente')

## 1. Carga de Datos IBWC

Sube los 3 CSVs principales:
- `Total Storage` (almacenamiento diario)
- `Change-in-Storage` (cambio neto diario)
- `Discharge.Best Available` (caudal aguas abajo, 15 min)

Y opcionalmente:
- `Percentage.Conservation` (para encontrar la ventana más crítica)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPCIÓN A: Subir archivos desde tu computadora (Colab)
# ═══════════════════════════════════════════════════════════════
try:
    from google.colab import files
    print('📁 Sube los archivos CSV del hackathon.')
    print('   Se necesitan al menos 3 archivos:')
    print('   1. Total Storage (Web-Daily)')
    print('   2. Discharge Total.Change-in-Storage')
    print('   3. Discharge.Best Available @08461300')
    print()
    uploaded = files.upload()
    UPLOADED_FILES = list(uploaded.keys())
    print(f'\n✅ {len(UPLOADED_FILES)} archivos subidos:')
    for f in UPLOADED_FILES:
        print(f'   • {f}')
    IN_COLAB = True
except ImportError:
    # No estamos en Colab — usar rutas locales
    IN_COLAB = False
    print('ℹ️ No estás en Colab. Se usarán rutas locales.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPCIÓN B: Si ya tienes los archivos en Google Drive
# ═══════════════════════════════════════════════════════════════
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/Hack Cuantico'  # Ajusta la ruta

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE RUTAS
# ═══════════════════════════════════════════════════════════════

import glob, os

def find_file(pattern, search_dirs=['.', '/content']):
    """Busca un archivo que contenga el patrón en el nombre."""
    for d in search_dirs:
        for f in glob.glob(os.path.join(d, '*')):
            if pattern.lower() in os.path.basename(f).lower():
                return f
    raise FileNotFoundError(f'No se encontró archivo con patrón: {pattern}')

# Auto-detectar archivos
try:
    STORAGE_FILE    = find_file('Total Storage.Web-Daily')
    DELTA_S_FILE    = find_file('Change-in-Storage')
    DISCHARGE_FILE  = find_file('Discharge.Best Available')
    print('✅ Archivos encontrados automáticamente:')
    print(f'   Storage:  {os.path.basename(STORAGE_FILE)}')
    print(f'   ΔStorage: {os.path.basename(DELTA_S_FILE)}')
    print(f'   Discharge: {os.path.basename(DISCHARGE_FILE)}')
except FileNotFoundError as e:
    print(f'❌ {e}')
    print('\n👉 Ingresa las rutas manualmente en la siguiente celda.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 👇 SI LA AUTO-DETECCIÓN FALLÓ, DESCOMENTA Y AJUSTA ESTAS RUTAS
# ═══════════════════════════════════════════════════════════════
# STORAGE_FILE   = 'DataSetExport-Total Storage.Web-Daily-ac-ft@08461200-Instantaneous-TCM-20260622185130.csv'
# DELTA_S_FILE   = 'DataSetExport-Discharge Total.Change-in-Storage@08461200-Instantaneous-TCM-20260622185956.csv'
# DISCHARGE_FILE = 'DataSetExport-Discharge.Best Available@08461300-Instantaneous-m^3 s-20260622190542.csv'

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CARGA Y PREPROCESAMIENTO
# ═══════════════════════════════════════════════════════════════

def load_ibwc_csv(filepath, value_col='value'):
    """Carga un CSV de IBWC, salta el header y limpia el disclaimer final."""
    df = pd.read_csv(filepath, skiprows=1)
    df.columns = ['timestamp', value_col, 'grade', 'approval', 'interp']
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    df = df.dropna(subset=['timestamp', value_col])
    df = df.sort_values('timestamp').reset_index(drop=True)
    return df

print('📂 Cargando datos IBWC...')

# CSV #1: Total Storage (diario, TCM)
storage_df = load_ibwc_csv(STORAGE_FILE, 'storage_tcm')
print(f'   Total Storage: {len(storage_df):,} registros '
      f'({storage_df.timestamp.min().date()} → {storage_df.timestamp.max().date()})')

# CSV #2: Change in Storage (diario, TCM)
delta_s_df = load_ibwc_csv(DELTA_S_FILE, 'delta_S_tcm')
print(f'   ΔStorage: {len(delta_s_df):,} registros '
      f'({delta_s_df.timestamp.min().date()} → {delta_s_df.timestamp.max().date()})')

# CSV #3: Discharge aguas abajo (m³/s)
release_df = load_ibwc_csv(DISCHARGE_FILE, 'discharge_m3s')
print(f'   Discharge: {len(release_df):,} registros '
      f'({release_df.timestamp.min().date()} → {release_df.timestamp.max().date()})')

print('\n✅ Datos cargados correctamente')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# RESAMPLEO A FRECUENCIA SEMANAL
# ═══════════════════════════════════════════════════════════════

print('🔄 Resampleando a frecuencia semanal...')

# Discharge: m³/s → TCM/semana = promedio_semanal × 7 × 86400 / 1000
release_weekly = (
    release_df.set_index('timestamp')['discharge_m3s']
    .resample('W').mean() * 7 * 86400 / 1000
).reset_index().rename(columns={'discharge_m3s': 'R_obs_tcm'}).dropna()

# Delta Storage: suma semanal
delta_s_weekly = (
    delta_s_df.set_index('timestamp')['delta_S_tcm']
    .resample('W').sum()
).reset_index().dropna()

# Merge por semana
weekly = pd.merge(release_weekly, delta_s_weekly, on='timestamp', how='inner')
print(f'   Semanas de datos combinados: {len(weekly)}')

# Buscar ventana más crítica: donde el storage promedio fue mínimo
storage_weekly_avg = (
    storage_df.set_index('timestamp')['storage_tcm']
    .resample('W').mean().dropna()
)

best_start = 0
min_avg_storage = float('inf')
s_vals = storage_weekly_avg.values
s_idx = storage_weekly_avg.index

for i in range(len(s_vals) - 26):
    avg = np.mean(s_vals[i:i+26])
    if avg < min_avg_storage:
        min_avg_storage = avg
        best_start = i

critical_start = s_idx[best_start]
critical_end = s_idx[best_start + 25]
print(f'   Ventana más crítica (26 sem): {critical_start.date()} → {critical_end.date()}')
print(f'   Storage promedio en ventana: {min_avg_storage:,.0f} TCM')

# Seleccionar datos de la ventana
mask = (weekly['timestamp'] >= critical_start) & (weekly['timestamp'] <= critical_end)
df_window = weekly[mask].reset_index(drop=True)

if len(df_window) < 26:
    print(f'   ⚠ Solo {len(df_window)} semanas combinadas en ventana óptima.')
    print('   → Usando últimas 52 semanas disponibles, tomando 26.')
    df_window = weekly.tail(52).head(26).reset_index(drop=True)

# Extraer arrays
R_obs_full = df_window['R_obs_tcm'].values
dS_obs_full = df_window['delta_S_tcm'].values
T_available = len(R_obs_full)
print(f'   Semanas seleccionadas: {T_available}')
print('\n✅ Preprocesamiento completo')

## 2. Parámetros Oficiales del Benchmark

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PARÁMETROS OFICIALES (del PDF FalconChallenge_V6)
# ═══════════════════════════════════════════════════════════════

S_MAX = storage_df['storage_tcm'].max()          # Capacidad total de conservación
S_MIN = 0.25 * S_MAX                              # Umbral crítico (eq. 15)
eta   = 0.10                                      # Restricción anti-hoarding

# Storage inicial: valor en la primera semana de la ventana
window_start = df_window['timestamp'].iloc[0]
s_at_start = storage_df[storage_df['timestamp'] <= window_start]['storage_tcm']
S_INIT = s_at_start.iloc[-1] if len(s_at_start) > 0 else S_MAX * 0.30

# Escala de ajuste (eq. 10-11)
R_median = np.median(R_obs_full)
delta_u  = 0.25 * R_median
u_max    = 2 * delta_u

# Niveles de ajuste (eq. 9)
levels_5 = np.array([-2, -1, 0, 1, 2]) * delta_u  # L=5
levels_3 = np.array([-2, 0, 2]) * delta_u           # L=3 (Small)

def get_weights(T, u_max, S_min):
    """Calcula pesos oficiales del benchmark."""
    w1 = 1.0 / ((T + 1) * S_min**2)
    w2 = 0.1 / (T * u_max**2)
    w3 = 0.1 / ((T - 1) * (2 * u_max)**2)
    return w1, w2, w3

# Instancias del benchmark
INSTANCES = {
    'Small':  {'T': 12, 'L': 3},
    'Medium': {'T': 26, 'L': 5},  # ← BENCHMARK OFICIAL
    'Large':  {'T': 52, 'L': 5},
}

print('╔══════════════════════════════════════════════════╗')
print('║       PARÁMETROS OFICIALES DEL BENCHMARK        ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  S_max      = {S_MAX:>14,.0f} TCM                 ║')
print(f'║  S_min      = {S_MIN:>14,.0f} TCM (25% S_max)     ║')
print(f'║  S_init     = {S_INIT:>14,.0f} TCM ({S_INIT/S_MAX*100:.1f}%)       ║')
print(f'║  Δu         = {delta_u:>14,.0f} TCM/sem             ║')
print(f'║  u_max      = {u_max:>14,.0f} TCM/sem             ║')
print(f'║  η          = {eta}                               ║')
print(f'║  R_obs med  = {R_median:>14,.0f} TCM/sem             ║')
print('╠══════════════════════════════════════════════════╣')
print('║  Niveles L=5: ', np.round(levels_5, 0), '       ║')
print('╚══════════════════════════════════════════════════╝')

## 3. Función SRS y Baselines

El **Storage Resilience Score** penaliza 3 cosas:
- `C_crit`: Caer por debajo del umbral crítico S_min
- `C_dev`: Desviarse mucho de la operación histórica
- `C_smooth`: Hacer cambios bruscos semana a semana

$$\text{SRS} = -(w_1 \cdot C_{crit} + w_2 \cdot C_{dev} + w_3 \cdot C_{smooth})$$

In [ ]:
def compute_SRS(u_seq, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta=0.10):
    """
    Calcula el Storage Resilience Score completo.
    
    Args:
        u_seq:  array de ajustes u(t) para cada semana
        S0:     almacenamiento inicial
        dS_obs: cambio de almacenamiento observado (semanal)
        R_obs:  liberación observada (semanal, TCM)
        S_min:  umbral crítico
        S_max:  capacidad máxima
        u_max:  ajuste máximo permitido
        w1-w3:  pesos oficiales
        eta:    límite anti-hoarding
    
    Returns:
        SRS, storages[], feasible, components{}
    """
    T = len(u_seq)
    S = S0
    C_crit, C_dev, C_smooth = 0.0, 0.0, 0.0
    feasible = True
    violations = []
    storages = [S0]

    for t in range(T):
        u = u_seq[t]
        R = R_obs[t] + u
        S_new = S + dS_obs[t] - u

        # Verificar restricciones
        if R < -1e-6:
            feasible = False
            violations.append(f't={t}: R={R:.0f} < 0')
        if abs(u) > u_max + 1e-6:
            feasible = False
            violations.append(f't={t}: |u|={abs(u):.0f} > u_max')
        if S_new < -1e-6:
            feasible = False
            violations.append(f't={t}: S={S_new:.0f} < 0')
        if S_new > S_max + 1e-6:
            feasible = False
            violations.append(f't={t}: S={S_new:.0f} > S_max')

        C_crit  += max(0, S_min - S_new) ** 2
        C_dev   += u ** 2
        if t > 0:
            C_smooth += (u - u_seq[t-1]) ** 2

        S = S_new
        storages.append(S)

    # Restricción anti-hoarding
    total_R = sum(R_obs[:T])
    if total_R > 0 and abs(sum(u_seq)) > eta * total_R:
        feasible = False
        violations.append(f'Anti-hoarding: |Σu|={abs(sum(u_seq)):.0f} > η·ΣR={eta*total_R:.0f}')

    SRS = -(w1 * C_crit + w2 * C_dev + w3 * C_smooth)

    components = {
        'C_crit': C_crit, 'C_dev': C_dev, 'C_smooth': C_smooth,
        'w1_C_crit': w1 * C_crit, 'w2_C_dev': w2 * C_dev, 'w3_C_smooth': w3 * C_smooth,
        'violations': violations,
    }
    return SRS, storages, feasible, components

print('✅ Función SRS definida')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# BASELINES CLÁSICOS
# ═══════════════════════════════════════════════════════════════

def threshold_rule(S0, dS_obs, R_obs, S_min, delta_u, T):
    """Baseline 2: reduce liberación cuando S < S_min."""
    u_seq, S = [], S0
    for t in range(T):
        adj = -delta_u if S < S_min else 0.0
        u_seq.append(adj)
        S = S + dS_obs[t] - adj
    return np.array(u_seq)


def adaptive_rule(S0, dS_obs, R_obs, S_min, S_max, delta_u, T, levels):
    """
    Baseline 3 (avanzado): regla adaptativa con múltiples umbrales.
    
    - S < 15% S_max → reducir 2Δu
    - S < S_min (25%) → reducir 1Δu
    - S > 75% S_max → aumentar 1Δu
    - Else → u = 0
    """
    u_seq, S = [], S0
    for t in range(T):
        if S < 0.15 * S_max:
            adj = -2 * delta_u
        elif S < S_min:
            adj = -1 * delta_u
        elif S > 0.75 * S_max:
            adj = +1 * delta_u
        else:
            adj = 0.0
        if R_obs[t] + adj < 0:
            adj = -R_obs[t]
        u_seq.append(adj)
        S = S + dS_obs[t] - adj
    return np.array(u_seq)

print('✅ Baselines definidos')

## 4. Simulated Annealing Clásico (Baseline Fuerte)

In [ ]:
def classical_SA(dS_obs, R_obs, S0, S_min, S_max, u_max,
                 w1, w2, w3, levels, T,
                 n_iter=100000, T_init=1.0, T_final=0.001,
                 seed=42, eta=0.10):
    """
    Simulated Annealing clásico directamente sobre el espacio de niveles.
    No usa QUBO; opera en el espacio discreto de L^T posibilidades.
    """
    np.random.seed(seed)
    L = len(levels)

    # Inicio: todo en nivel 0 (historical replay)
    current = np.full(T, L // 2, dtype=int)
    current_u = levels[current]
    current_srs, _, current_feas, _ = compute_SRS(
        current_u, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)

    best = current.copy()
    best_srs = current_srs
    best_u = current_u.copy()
    history = [best_srs]

    for i in range(n_iter):
        temp = T_init * (T_final / T_init) ** (i / n_iter)

        # Vecino: cambiar 1 semana a un nivel aleatorio
        neighbor = current.copy()
        t_change = np.random.randint(T)
        neighbor[t_change] = np.random.randint(L)
        neighbor_u = levels[neighbor]

        neighbor_srs, _, neighbor_feas, _ = compute_SRS(
            neighbor_u, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta)

        if not neighbor_feas:
            neighbor_srs -= 1e6  # penalizar infactibilidad

        delta = neighbor_srs - current_srs
        if delta > 0 or np.random.random() < np.exp(delta / max(temp, 1e-15)):
            current = neighbor
            current_u = neighbor_u
            current_srs = neighbor_srs

            if current_srs > best_srs:
                best = current.copy()
                best_srs = current_srs
                best_u = current_u.copy()

        if i % 1000 == 0:
            history.append(best_srs)

    return best_u, best_srs, history

print('✅ Classical SA definido')

## 5. Formulación QUBO

La formulación usa **one-hot encoding**:
- Variable $x_{t,k} \in \{0,1\}$: vale 1 si en la semana $t$ se elige el nivel $k$
- Total de variables: $N = T \times L$
- Restricción: exactamente un $x_{t,k} = 1$ por cada semana $t$

La matriz QUBO codifica:
1. **Penalización one-hot** (λ grande)
2. **C_dev**: penaliza ajustes grandes
3. **C_smooth**: penaliza cambios bruscos entre semanas
4. **C_crit**: penalización linealizada por caer bajo S_min
5. **Anti-hoarding**: penalización del balance total de ajustes

In [ ]:
def build_qubo(dS_obs, R_obs, S0, S_min, S_max, u_max,
               w1, w2, w3, levels, T,
               lambda_oh=None, lambda_crit=None, eta=0.10):
    """
    Construye la matriz QUBO con one-hot encoding.
    
    Variables: x[t,k] ∈ {0,1},  k=0..L-1  para cada t=0..T-1
    Total: N = T × L variables binarias
    
    Returns: Q (dict sparse para dimod), N (número de variables)
    """
    L = len(levels)
    N = T * L
    Q = {}  # formato sparse dict {(i,j): coeff}

    def idx(t, k):
        return t * L + k

    # Auto-escala de penalizaciones
    if lambda_oh is None:
        obj_scale = max(abs(w2 * u_max**2), abs(w3 * (2*u_max)**2))
        lambda_oh = max(1e4 * obj_scale, 1.0)
    if lambda_crit is None:
        lambda_crit = w1

    # ─── 1. Penalización One-Hot ───
    # (Σ_k x[t,k] - 1)² = Σ_k x²[t,k] - 2·Σ_k x[t,k] + 1 + 2·Σ_{k<k'} x[t,k]·x[t,k']
    # Dado que x²=x para binarios: coeff lineal = -1, coeff cuadrático = +2
    for t in range(T):
        for k in range(L):
            i = idx(t, k)
            Q[(i, i)] = Q.get((i, i), 0) + lambda_oh * (-1)
            for k2 in range(k + 1, L):
                j = idx(t, k2)
                key = (min(i, j), max(i, j))
                Q[key] = Q.get(key, 0) + lambda_oh * 2

    # ─── 2. C_dev: Σ_t u(t)² → diagonal ───
    for t in range(T):
        for k in range(L):
            i = idx(t, k)
            Q[(i, i)] = Q.get((i, i), 0) + w2 * levels[k]**2

    # ─── 3. C_smooth: Σ_t (u(t) - u(t-1))² ───
    for t in range(1, T):
        for k in range(L):
            for k2 in range(L):
                i = idx(t, k)
                j = idx(t - 1, k2)
                contrib = w3 * (levels[k] - levels[k2])**2
                if i == j:
                    Q[(i, i)] = Q.get((i, i), 0) + contrib
                else:
                    key = (min(i, j), max(i, j))
                    Q[key] = Q.get(key, 0) + contrib

    # ─── 4. C_crit (linealizada) ───
    S_baseline = np.zeros(T + 1)
    S_baseline[0] = S0
    for t in range(T):
        S_baseline[t + 1] = S_baseline[t] + dS_obs[t]

    for t in range(T):
        for k in range(L):
            i = idx(t, k)
            penalty = 0.0
            for t2 in range(t, T):
                S_no_adj = S_baseline[t2 + 1]
                if S_no_adj < S_min:
                    gap = S_min - S_no_adj
                    penalty += lambda_crit * 2 * gap * levels[k]
            Q[(i, i)] = Q.get((i, i), 0) + penalty

    # ─── 5. Anti-hoarding ───
    lambda_ah = w2 * 0.1
    for t1 in range(T):
        for k1 in range(L):
            i = idx(t1, k1)
            for t2 in range(t1, T):
                for k2 in range(L):
                    j = idx(t2, k2)
                    contrib = lambda_ah * levels[k1] * levels[k2]
                    if i == j:
                        Q[(i, i)] = Q.get((i, i), 0) + contrib
                    else:
                        key = (min(i, j), max(i, j))
                        Q[key] = Q.get(key, 0) + contrib

    return Q, N


def decode_qubo(sample, T, L, levels):
    """Decodifica solución one-hot → secuencia u(t)."""
    u_seq = []
    for t in range(T):
        chosen = [k for k in range(L) if sample.get(t * L + k, 0) == 1]
        if len(chosen) == 1:
            u_seq.append(levels[chosen[0]])
        elif len(chosen) == 0:
            u_seq.append(0.0)
        else:
            u_seq.append(levels[min(chosen, key=lambda k: abs(levels[k]))])
    return np.array(u_seq)


def repair_feasibility(u_seq, R_obs, S0, dS_obs, S_min, S_max, u_max, eta, levels):
    """Repara soluciones infactibles con greedy post-processing."""
    T = len(u_seq)
    u_rep = u_seq.copy()

    # Clampear y snap a niveles
    for t in range(T):
        u_rep[t] = np.clip(u_rep[t], -u_max, u_max)
        u_rep[t] = levels[np.argmin(np.abs(levels - u_rep[t]))]

    # Asegurar R(t) >= 0 y 0 <= S(t) <= S_max
    S = S0
    for t in range(T):
        if R_obs[t] + u_rep[t] < 0:
            valid = levels[levels >= -R_obs[t] - 1e-6]
            u_rep[t] = valid[0] if len(valid) > 0 else levels[0]

        S_new = S + dS_obs[t] - u_rep[t]
        if S_new < 0:
            candidates = levels[levels < u_rep[t]]
            if len(candidates) > 0:
                u_rep[t] = candidates[-1]
        elif S_new > S_max:
            candidates = levels[levels > u_rep[t]]
            if len(candidates) > 0:
                u_rep[t] = candidates[0]

        S = S + dS_obs[t] - u_rep[t]

    # Anti-hoarding
    total_R = sum(R_obs[:T])
    total_u = sum(u_rep)
    if total_R > 0 and abs(total_u) > eta * total_R:
        excess = abs(total_u) - eta * total_R
        sign = np.sign(total_u)
        for i in np.argsort(-np.abs(u_rep)):
            if excess <= 0:
                break
            old = u_rep[i]
            if sign > 0 and old > 0:
                u_rep[i] = levels[np.argmin(np.abs(levels - max(0, old - excess)))]
                excess -= (old - u_rep[i])
            elif sign < 0 and old < 0:
                u_rep[i] = levels[np.argmin(np.abs(levels - min(0, old + excess)))]
                excess -= (u_rep[i] - old)

    return u_rep

print('✅ QUBO builder y funciones auxiliares definidas')

## 6. Ejecución: Todas las Instancias

Ejecutamos todos los métodos (baselines + SA clásico + QUBO) para cada instancia del benchmark.

In [ ]:
results = {}

for inst_name, inst_cfg in INSTANCES.items():
    T = min(inst_cfg['T'], T_available)
    L = inst_cfg['L']
    inst_levels = levels_3 if L == 3 else levels_5

    R_T  = R_obs_full[:T]
    dS_T = dS_obs_full[:T]
    w1, w2, w3 = get_weights(T, u_max, S_MIN)
    args = (S_INIT, dS_T, R_T, S_MIN, S_MAX, u_max, w1, w2, w3)

    print(f'\n{"="*60}')
    print(f'  INSTANCIA: {inst_name} (T={T}, L={L}, N={T*L} vars)')
    print(f'{"="*60}')

    # ── Baselines ──
    u_hist = np.zeros(T)
    SRS_hist, S_hist, feas_hist, comp_hist = compute_SRS(u_hist, *args)

    u_rule = threshold_rule(S_INIT, dS_T, R_T, S_MIN, delta_u, T)
    SRS_rule, S_rule, feas_rule, comp_rule = compute_SRS(u_rule, *args)

    u_adapt = adaptive_rule(S_INIT, dS_T, R_T, S_MIN, S_MAX, delta_u, T, inst_levels)
    SRS_adapt, S_adapt, feas_adapt, comp_adapt = compute_SRS(u_adapt, *args)

    # ── Classical SA ──
    print(f'  🔥 Simulated Annealing clásico...')
    t0 = time.time()
    u_sa, _, sa_history = classical_SA(
        dS_T, R_T, S_INIT, S_MIN, S_MAX, u_max, w1, w2, w3,
        inst_levels, T, n_iter=100000, seed=42)
    time_sa = time.time() - t0
    SRS_sa, S_sa, feas_sa, comp_sa = compute_SRS(u_sa, *args)

    # ── QUBO + SA (dimod/neal) ──
    print(f'  ⚛️  Construyendo QUBO ({T*L}×{T*L})...')
    t0 = time.time()
    Q_dict, N_vars = build_qubo(
        dS_T, R_T, S_INIT, S_MIN, S_MAX, u_max, w1, w2, w3,
        inst_levels, T)
    time_build = time.time() - t0

    print(f'     QUBO construido en {time_build:.2f}s ({len(Q_dict):,} entradas no-cero)')
    print(f'  ⚛️  Resolviendo con SA (dimod)...')

    bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict)
    sampler = neal.SimulatedAnnealingSampler()

    t0 = time.time()
    response = sampler.sample(bqm, num_reads=200, num_sweeps=5000, seed=42)
    time_solve = time.time() - t0

    u_qubo_raw = decode_qubo(response.first.sample, T, L, inst_levels)
    u_qubo = repair_feasibility(u_qubo_raw, R_T, S_INIT, dS_T, S_MIN, S_MAX, u_max, eta, inst_levels)
    SRS_qubo, S_qubo, feas_qubo, comp_qubo = compute_SRS(u_qubo, *args)

    # ── Resultados ──
    print(f'\n  {"Método":<28} {"SRS":>12} {"ΔSRS":>12} {"Factible":>9} {"Tiempo":>8}')
    print(f'  {"-"*72}')
    for name, srs, feas, t_exec in [
        ('Historical Replay', SRS_hist, feas_hist, 0),
        ('Threshold Rule', SRS_rule, feas_rule, 0),
        ('Adaptive Rule', SRS_adapt, feas_adapt, 0),
        ('Classical SA', SRS_sa, feas_sa, time_sa),
        ('QUBO + SA (dimod)', SRS_qubo, feas_qubo, time_build + time_solve),
    ]:
        d = srs - SRS_hist
        f_icon = '✅' if feas else '❌'
        print(f'  {name:<28} {srs:>12.6f} {d:>+12.6f} {f_icon:>9} {t_exec:>7.2f}s')

    results[inst_name] = {
        'T': T, 'L': L,
        'SRS_hist': SRS_hist, 'S_hist': S_hist, 'comp_hist': comp_hist,
        'SRS_rule': SRS_rule, 'S_rule': S_rule, 'feas_rule': feas_rule, 'comp_rule': comp_rule,
        'SRS_adapt': SRS_adapt, 'S_adapt': S_adapt, 'feas_adapt': feas_adapt, 'comp_adapt': comp_adapt,
        'SRS_sa': SRS_sa, 'S_sa': S_sa, 'feas_sa': feas_sa, 'comp_sa': comp_sa,
        'u_hist': u_hist, 'u_rule': u_rule, 'u_adapt': u_adapt, 'u_sa': u_sa,
        'SRS_qubo': SRS_qubo, 'S_qubo': S_qubo, 'feas_qubo': feas_qubo, 'comp_qubo': comp_qubo,
        'u_qubo': u_qubo,
        'sa_history': sa_history,
        'R_T': R_T, 'dS_T': dS_T,
        'w1': w1, 'w2': w2, 'w3': w3,
        'time_sa': time_sa, 'time_qubo_build': time_build, 'time_qubo_solve': time_solve,
    }

print('\n\n✅ Todas las instancias procesadas')

## 7. Gráfica 1 — Almacenamiento Histórico del Embalse Falcon

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(storage_df['timestamp'], storage_df['storage_tcm'],
        color=COLORS['sa'], linewidth=0.8, alpha=0.9)
ax.axhline(y=S_MIN, color=COLORS['smin'], linewidth=2, linestyle=':',
           label=f'S_min = {S_MIN:,.0f} TCM (25%)')
ax.axhline(y=S_MAX, color=COLORS['smax'], linewidth=1.5, linestyle=':', alpha=0.5,
           label=f'S_max = {S_MAX:,.0f} TCM')
ax.fill_between(storage_df['timestamp'], 0, S_MIN, alpha=0.05, color=COLORS['smin'])

ax.set_xlabel('Fecha')
ax.set_ylabel('Almacenamiento (TCM)')
ax.set_title('Embalse Internacional Falcon — Almacenamiento Histórico (1953–presente)',
             fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Gráfica 2 — Trayectorias de Almacenamiento (Medium)

In [ ]:
def plot_trajectories(inst_name):
    r = results[inst_name]
    T = r['T']
    weeks = np.arange(T + 1)

    fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                              gridspec_kw={'height_ratios': [3, 1]})

    # ── Storage ──
    ax = axes[0]
    ax.fill_between(weeks, 0, r['S_hist'], alpha=0.05, color=COLORS['hist'])
    ax.plot(weeks, r['S_hist'], color=COLORS['hist'], lw=2,
            label=f'Historical (SRS={r["SRS_hist"]:.4f})', ls='--')
    ax.plot(weeks, r['S_rule'], color=COLORS['rule'], lw=2.5,
            label=f'Threshold (SRS={r["SRS_rule"]:.4f})')
    ax.plot(weeks, r['S_adapt'], color=COLORS['adapt'], lw=2.5,
            label=f'Adaptive (SRS={r["SRS_adapt"]:.4f})')
    ax.plot(weeks, r['S_sa'], color=COLORS['sa'], lw=2.5,
            label=f'Classical SA (SRS={r["SRS_sa"]:.4f})')
    ax.plot(weeks, r['S_qubo'], color=COLORS['qubo'], lw=3,
            label=f'QUBO+SA (SRS={r["SRS_qubo"]:.4f})', marker='o', ms=4)

    ax.axhline(y=S_MIN, color=COLORS['smin'], lw=2, ls=':', alpha=0.8,
               label=f'S_min ({S_MIN:,.0f} TCM)')
    ax.fill_between(weeks, 0, S_MIN, alpha=0.08, color=COLORS['smin'])
    ax.text(T*0.02, S_MIN*0.5, 'ZONA CRÍTICA', color=COLORS['smin'],
            fontsize=10, fontweight='bold', alpha=0.6)

    ax.set_ylabel('Almacenamiento (TCM)')
    ax.set_title(f'Trayectorias de Almacenamiento — {inst_name} (T={T})',
                 fontsize=16, fontweight='bold', pad=15)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
    ax.set_xlim(0, T)
    ax.grid(True, alpha=0.3)

    # ── Adjustments u(t) ──
    ax2 = axes[1]
    wk = np.arange(T)
    w = 0.2
    ax2.bar(wk - 1.5*w, r['u_rule'], w, color=COLORS['rule'], alpha=0.8, label='Threshold')
    ax2.bar(wk - 0.5*w, r['u_adapt'], w, color=COLORS['adapt'], alpha=0.8, label='Adaptive')
    ax2.bar(wk + 0.5*w, r['u_sa'], w, color=COLORS['sa'], alpha=0.8, label='Classical SA')
    ax2.bar(wk + 1.5*w, r['u_qubo'], w, color=COLORS['qubo'], alpha=0.8, label='QUBO+SA')
    ax2.axhline(y=0, color='#c9d1d9', lw=0.5, alpha=0.5)
    ax2.axhline(y=u_max, color=COLORS['smin'], lw=1, ls=':', alpha=0.5)
    ax2.axhline(y=-u_max, color=COLORS['smin'], lw=1, ls=':', alpha=0.5)
    ax2.set_ylabel('u(t) Ajuste (TCM)')
    ax2.set_xlabel('Semana')
    ax2.legend(loc='lower right', fontsize=8, ncol=4)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Mostrar Medium (benchmark oficial)
plot_trajectories('Medium')

## 9. Gráfica 3 — Trayectorias Small y Large

In [ ]:
plot_trajectories('Small')

In [ ]:
if 'Large' in results:
    plot_trajectories('Large')

## 10. Gráfica 4 — Comparación SRS entre Métodos e Instancias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

instances = list(results.keys())
methods = ['Historical', 'Threshold', 'Adaptive', 'Classical SA', 'QUBO+SA']
colors_m = [COLORS['hist'], COLORS['rule'], COLORS['adapt'], COLORS['sa'], COLORS['qubo']]
keys_m = ['SRS_hist', 'SRS_rule', 'SRS_adapt', 'SRS_sa', 'SRS_qubo']

x = np.arange(len(instances))
w = 0.15

# Panel izq: SRS absoluto
for i, (method, key, color) in enumerate(zip(methods, keys_m, colors_m)):
    vals = [results[inst].get(key, 0) for inst in instances]
    axes[0].bar(x + i*w - 2*w, vals, w, label=method, color=color, alpha=0.85)

axes[0].set_xticks(x)
axes[0].set_xticklabels(instances)
axes[0].set_ylabel('SRS (mayor = mejor)')
axes[0].set_title('SRS por Método e Instancia', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, axis='y', alpha=0.3)

# Panel der: ΔSRS vs Historical
for i, (method, key, color) in enumerate(zip(methods[1:], keys_m[1:], colors_m[1:])):
    vals = [results[inst].get(key, 0) - results[inst]['SRS_hist'] for inst in instances]
    axes[1].bar(x + i*w - 1.5*w, vals, w, label=method, color=color, alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels(instances)
axes[1].set_ylabel('ΔSRS vs Historical Replay')
axes[1].set_title('Mejora sobre Historical Replay', fontsize=14, fontweight='bold')
axes[1].axhline(y=0, color='#c9d1d9', lw=0.5, alpha=0.5)
axes[1].legend(fontsize=9)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Gráfica 5 — Descomposición del Costo

In [ ]:
def plot_cost_decomposition(inst_name):
    r = results[inst_name]
    fig, ax = plt.subplots(figsize=(12, 7))

    methods = ['Historical', 'Threshold', 'Adaptive', 'Classical SA', 'QUBO+SA']
    comps = [r['comp_hist'], r['comp_rule'], r['comp_adapt'], r['comp_sa'], r['comp_qubo']]

    x = np.arange(len(methods))
    w = 0.25

    ax.bar(x - w, [c['w1_C_crit'] for c in comps], w,
           label='w₁·C_crit (pen. crítica)', color='#f85149', alpha=0.85)
    ax.bar(x,     [c['w2_C_dev'] for c in comps], w,
           label='w₂·C_dev (desviación)', color='#58a6ff', alpha=0.85)
    ax.bar(x + w, [c['w3_C_smooth'] for c in comps], w,
           label='w₃·C_smooth (suavidad)', color='#d2a8ff', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylabel('Costo Ponderado (menor = mejor)')
    ax.set_title(f'Descomposición del Costo — {inst_name}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_cost_decomposition('Medium')

## 12. Gráfica 6 — Convergencia del Simulated Annealing

In [ ]:
def plot_convergence(inst_name):
    r = results[inst_name]
    fig, ax = plt.subplots(figsize=(12, 6))

    history = r['sa_history']
    iters = np.arange(len(history)) * 1000

    ax.plot(iters, history, color=COLORS['sa'], lw=2, label='Classical SA')
    ax.axhline(y=r['SRS_hist'], color=COLORS['hist'], lw=1.5, ls='--',
               label=f'Historical: {r["SRS_hist"]:.4f}', alpha=0.7)
    ax.axhline(y=r['SRS_rule'], color=COLORS['rule'], lw=1.5, ls='--',
               label=f'Threshold: {r["SRS_rule"]:.4f}', alpha=0.7)
    ax.axhline(y=r['SRS_qubo'], color=COLORS['qubo'], lw=2, ls='-.',
               label=f'QUBO+SA: {r["SRS_qubo"]:.4f}')

    ax.set_xlabel('Iteraciones')
    ax.set_ylabel('Mejor SRS')
    ax.set_title(f'Convergencia del SA — {inst_name}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_convergence('Medium')

## 13. Gráfica 7 — Perfil de Liberaciones

In [ ]:
def plot_releases(inst_name):
    r = results[inst_name]
    T = r['T']
    weeks = np.arange(T)
    R_obs_plot = r['R_T']

    fig, ax = plt.subplots(figsize=(14, 6))

    ax.fill_between(weeks, 0, R_obs_plot, alpha=0.15, color=COLORS['hist'])
    ax.plot(weeks, R_obs_plot, color=COLORS['hist'], lw=2,
            label='R_obs (histórico)', ls='--')
    ax.plot(weeks, R_obs_plot + r['u_qubo'], color=COLORS['qubo'], lw=2.5,
            label='R_opt QUBO+SA', marker='o', ms=4)
    ax.plot(weeks, R_obs_plot + r['u_sa'], color=COLORS['sa'], lw=2,
            label='R_opt Classical SA', alpha=0.8)

    ax.axhline(y=0, color=COLORS['smin'], lw=1, alpha=0.3)
    ax.set_xlabel('Semana')
    ax.set_ylabel('Liberación (TCM/semana)')
    ax.set_title(f'Perfil de Liberaciones — {inst_name}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_releases('Medium')

## 14. Gráfica 8 — Estructura de la Matriz QUBO

In [ ]:
def plot_qubo_matrix(inst_name):
    r = results[inst_name]
    T, L = r['T'], r['L']
    inst_levels = levels_3 if L == 3 else levels_5
    w1, w2, w3 = r['w1'], r['w2'], r['w3']

    Q_dict, N = build_qubo(
        r['dS_T'], r['R_T'], S_INIT, S_MIN, S_MAX, u_max,
        w1, w2, w3, inst_levels, T)

    Q_dense = np.zeros((N, N))
    for (i, j), v in Q_dict.items():
        Q_dense[i, j] = v
        if i != j:
            Q_dense[j, i] = v

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    Q_log = np.sign(Q_dense) * np.log1p(np.abs(Q_dense))
    im = axes[0].imshow(Q_log, cmap='RdBu_r', aspect='equal', interpolation='nearest')
    axes[0].set_title(f'Matriz QUBO ({N}×{N}) — log scale', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Variable index')
    axes[0].set_ylabel('Variable index')
    plt.colorbar(im, ax=axes[0], shrink=0.8)
    for t in range(T):
        axes[0].axhline(y=t*L-0.5, color='white', lw=0.3, alpha=0.3)
        axes[0].axvline(x=t*L-0.5, color='white', lw=0.3, alpha=0.3)

    block = min(3*L, N)
    im2 = axes[1].imshow(Q_log[:block, :block], cmap='RdBu_r', aspect='equal', interpolation='nearest')
    axes[1].set_title(f'Zoom: Primeras 3 semanas ({block}×{block})', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Variable index')
    axes[1].set_ylabel('Variable index')
    plt.colorbar(im2, ax=axes[1], shrink=0.8)
    for t in range(4):
        axes[1].axhline(y=t*L-0.5, color='white', lw=1, alpha=0.5)
        axes[1].axvline(x=t*L-0.5, color='white', lw=1, alpha=0.5)

    plt.suptitle(f'Estructura de la Matriz QUBO — {inst_name}',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_qubo_matrix('Small')

In [ ]:
plot_qubo_matrix('Medium')

## 15. Gráfica 9 — Análisis de Escalado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

instances = list(results.keys())
T_vals = [results[i]['T'] for i in instances]
L_vals = [results[i]['L'] for i in instances]

# SRS por instancia
for method, key, color in [('Historical', 'SRS_hist', COLORS['hist']),
                           ('Threshold', 'SRS_rule', COLORS['rule']),
                           ('Classical SA', 'SRS_sa', COLORS['sa']),
                           ('QUBO+SA', 'SRS_qubo', COLORS['qubo'])]:
    vals = [results[i].get(key, 0) for i in instances]
    axes[0].plot(T_vals, vals, 'o-', color=color, lw=2, ms=8, label=method)

axes[0].set_xlabel('T (semanas)')
axes[0].set_ylabel('SRS')
axes[0].set_title('SRS vs Horizonte Temporal', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Espacio de búsqueda
search = [l**t for t, l in zip(T_vals, L_vals)]
axes[1].semilogy(T_vals, search, 'D-', color=COLORS['accent'], lw=2.5, ms=10)
for t, s, inst in zip(T_vals, search, instances):
    axes[1].annotate(f'{inst}\n{s:.1e}', (t, s), textcoords='offset points',
                     xytext=(0, 15), ha='center', fontsize=9, color=COLORS['accent'])
axes[1].set_xlabel('T (semanas)')
axes[1].set_ylabel('Espacio de búsqueda (L^T)')
axes[1].set_title('Escalado Exponencial', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Runtime
sa_t = [results[i]['time_sa'] for i in instances]
qubo_t = [results[i]['time_qubo_build'] + results[i]['time_qubo_solve'] for i in instances]
x = np.arange(len(instances))
axes[2].bar(x - 0.2, sa_t, 0.35, color=COLORS['sa'], alpha=0.85, label='Classical SA')
axes[2].bar(x + 0.2, qubo_t, 0.35, color=COLORS['qubo'], alpha=0.85, label='QUBO+SA')
axes[2].set_xticks(x)
axes[2].set_xticklabels(instances)
axes[2].set_ylabel('Tiempo (s)')
axes[2].set_title('Runtime por Instancia', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 16. Reporte Final — Tabla de Resultados

In [ ]:
print('╔' + '═'*76 + '╗')
print('║' + '  REPORTE FINAL — Challenge A: Resilient Release Scheduling'.center(76) + '║')
print('║' + '  Falcon Reservoir · OQI Quantum Hackathon · Junio 2026'.center(76) + '║')
print('╠' + '═'*76 + '╣')

for inst_name, r in results.items():
    print(f'║  {inst_name} (T={r["T"]}, L={r["L"]}, N={r["T"]*r["L"]} qubits)'.ljust(76) + ' ║')
    print('║  ' + '-'*72 + '  ║')
    print(f'║  {"Método":<26} {"SRS":>12} {"ΔSRS":>12} {"Factible":>9} {"Time(s)":>8}  ║')
    print('║  ' + '-'*72 + '  ║')

    for name, srs, feas, t_exec in [
        ('Historical Replay',  r['SRS_hist'],  True,           0),
        ('Threshold Rule',     r['SRS_rule'],  r['feas_rule'], 0),
        ('Adaptive Rule',      r['SRS_adapt'], r['feas_adapt'],0),
        ('Classical SA',       r['SRS_sa'],    r['feas_sa'],   r['time_sa']),
        ('QUBO + SA (dimod)',  r['SRS_qubo'],  r['feas_qubo'], r['time_qubo_build']+r['time_qubo_solve']),
    ]:
        d = srs - r['SRS_hist']
        f_icon = '✅' if feas else '❌'
        print(f'║  {name:<26} {srs:>12.6f} {d:>+12.6f} {f_icon:>9} {t_exec:>7.2f}s  ║')

    print('║' + ' '*76 + '║')

print('╠' + '═'*76 + '╣')
print(f'║  S_max = {S_MAX:>12,.0f} TCM   |   S_min = {S_MIN:>12,.0f} TCM'.ljust(76) + '  ║')
print(f'║  S_init= {S_INIT:>12,.0f} TCM   |   Δu    = {delta_u:>12,.0f} TCM/sem'.ljust(76) + '  ║')
print(f'║  η     = {eta}                |   u_max = {u_max:>12,.0f} TCM/sem'.ljust(76) + '  ║')
print('╚' + '═'*76 + '╝')

## 17. Exportar Solución (para entrega)

Exporta la secuencia óptima u(t) y la trayectoria S(t) como CSV.

In [ ]:
# Exportar solución Medium (benchmark oficial)
r = results['Medium']
T = r['T']

solution_df = pd.DataFrame({
    'week': range(T),
    'u_qubo': r['u_qubo'],
    'u_sa': r['u_sa'],
    'R_obs': r['R_T'],
    'R_opt_qubo': r['R_T'] + r['u_qubo'],
    'R_opt_sa': r['R_T'] + r['u_sa'],
    'S_hist': r['S_hist'][1:],
    'S_qubo': r['S_qubo'][1:],
    'S_sa': r['S_sa'][1:],
})

solution_df.to_csv('falcon_solution_medium.csv', index=False)
print('✅ Solución exportada: falcon_solution_medium.csv')
print()
display(solution_df.head(10))

# Descargar en Colab
try:
    from google.colab import files
    files.download('falcon_solution_medium.csv')
except:
    pass